# 🔗 Day 05: Union-Find (Disjoint Set Union) — Grouping Things Together

---

## What Is Union-Find?

A data structure that tracks elements partitioned into disjoint groups. Supports two operations:

- **Find(x):** Which group does x belong to? (returns root/representative)
- **Union(x, y):** Merge the groups containing x and y.

Think of it like **friend circles.** Everyone in a circle has one "representative." When two circles merge, one representative adopts the other.

```
Initially:  {A}  {B}  {C}  {D}  {E}
Union(A,B): {A,B} {C} {D} {E}
Union(C,D): {A,B} {C,D} {E}
Union(B,D): {A,B,C,D} {E}
Find(A) == Find(C)?  → YES (same group!)
```

### Two Key Optimizations

1. **Path Compression** (in `find`): Make every node point directly to root → nearly O(1) lookups
2. **Union by Rank** (in `union`): Attach shorter tree under taller tree → keeps tree flat

Together: **amortized O(α(n)) per operation** — effectively constant time!

---

In [ ]:
class UnionFind:
    """Standard Union-Find template — memorize this!"""
    
    def __init__(self, n):
        self.parent = list(range(n))  # Everyone is their own parent initially
        self.rank = [0] * n            # Height estimate
        self.count = n                 # Number of connected components
    
    def find(self, x):
        # Path compression: make x point directly to root
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py:
            return False  # Already in the same set
        # Union by rank: attach shorter tree under taller
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        self.count -= 1
        return True  # Merge happened
    
    def connected(self, x, y):
        return self.find(x) == self.find(y)

# Quick test
uf = UnionFind(5)
uf.union(0, 1)
uf.union(2, 3)
uf.union(1, 3)
print(f"0 and 3 connected? {uf.connected(0, 3)}")  # True
print(f"Components: {uf.count}")                     # 2 ({0,1,2,3} and {4})

---

## 🎯 Problem 1: Number of Connected Components (LC #323)

Given n nodes and edges, find the number of connected components.

**This is the classic Union-Find problem.** Just union all edges and return `uf.count`.

In [ ]:
def countComponents(n, edges):
    """
    Union all edges, return component count.
    Time: O(E * α(N)) ≈ O(E), Space: O(N)
    """
    uf = UnionFind(n)
    for u, v in edges:
        uf.union(u, v)
    return uf.count

print(countComponents(5, [[0,1],[1,2],[3,4]]))    # 2
print(countComponents(5, [[0,1],[1,2],[2,3],[3,4]]))  # 1

---

## 🎯 Problem 2: Redundant Connection (LC #684)

A tree with one extra edge. Find the edge that, if removed, makes it a tree again.

**Key Insight:** Process edges one by one. The FIRST edge where both nodes already belong to the same component is the answer — it's creating a cycle.

In [ ]:
def findRedundantConnection(edges):
    """
    The edge that creates a cycle = both endpoints already connected.
    Time: O(N * α(N)) ≈ O(N), Space: O(N)
    """
    n = len(edges)
    uf = UnionFind(n + 1)  # 1-indexed nodes
    
    for u, v in edges:
        if not uf.union(u, v):
            return [u, v]  # Already connected → this edge is redundant

print(findRedundantConnection([[1,2],[1,3],[2,3]]))          # [2,3]
print(findRedundantConnection([[1,2],[2,3],[3,4],[1,4],[1,5]]))  # [1,4]

---

## 🎯 Problem 3: Accounts Merge (LC #721)

Merge accounts that share an email. This is "group by common elements" — classic Union-Find.

**Approach:**
1. Map each email to an account index.
2. If an email appears in two accounts, union those accounts.
3. Group emails by their root account. Sort and format.

In [ ]:
def accountsMerge(accounts):
    """
    Union-Find on account indices, keyed by shared emails.
    Time: O(N * K * α(N)), Space: O(N * K) where K = avg emails per account
    """
    uf = UnionFind(len(accounts))
    email_to_idx = {}  # email → first account index that owns it
    
    # Step 1: Union accounts that share emails
    for i, account in enumerate(accounts):
        for email in account[1:]:
            if email in email_to_idx:
                uf.union(i, email_to_idx[email])
            else:
                email_to_idx[email] = i
    
    # Step 2: Group all emails by root account
    from collections import defaultdict
    groups = defaultdict(set)
    for email, idx in email_to_idx.items():
        root = uf.find(idx)
        groups[root].add(email)
    
    # Step 3: Format output
    return [[accounts[root][0]] + sorted(emails) for root, emails in groups.items()]

accounts = [
    ["John", "john@mail.com", "john_newyork@mail.com"],
    ["John", "john@mail.com", "john00@mail.com"],
    ["Mary", "mary@mail.com"],
    ["John", "johnnybravo@mail.com"]
]
for acc in accountsMerge(accounts):
    print(acc)

---

## 🗺️ Union-Find Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  UNION-FIND PATTERNS                                                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHEN TO USE:                                                       ║
║    • "Connected components" or "grouping"                           ║
║    • "Is X connected to Y?" queries (dynamic)                      ║
║    • Detecting cycles in undirected graphs                          ║
║    • Merging groups by shared property                              ║
║                                                                      ║
║  TEMPLATE: parent[] + find(path compress) + union(by rank)          ║
║  return False from union = CYCLE detected                           ║
║                                                                      ║
║  Union-Find vs BFS/DFS for components:                              ║
║    • Static graph → BFS/DFS is simpler                             ║
║    • Dynamic edges (added over time) → Union-Find wins             ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 323 | Number of Connected Components | Medium | Basic union + count |
| 684 | Redundant Connection | Medium | Cycle = union returns False |
| 721 | Accounts Merge | Medium | Group by shared property |
| 261 | Graph Valid Tree | Medium | n-1 edges + fully connected |
| 547 | Number of Provinces | Medium | Union-Find on adjacency matrix |
| 1319 | Number of Operations to Make Network Connected | Medium | Need n-1 edges minimum |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Union-Find = grouping things with near O(1) operations         │
├──────────────────────────────────────────────────────────────────────┤
│  📌 ALWAYS use path compression + union by rank together           │
├──────────────────────────────────────────────────────────────────────┤
│  📌 union() returns False → both already connected → CYCLE         │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Track .count for number of components                          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Shared property" problems → map property to index → union    │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Shortest Path

Dijkstra's algorithm — finding the cheapest route in weighted graphs.